<a href="https://colab.research.google.com/github/ZainDev04/Flyrank-ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import os
import sys


IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import files
    print("Please upload: data/raw/content_refresh_anonymized.csv")
    uploaded = files.upload()


    os.makedirs('data/raw', exist_ok=True)
    for filename in uploaded.keys():
        if 'content_refresh' in filename:
            import shutil
            shutil.move(filename, 'data/raw/content_refresh_anonymized.csv')
            print(f"Moved {filename} -> data/raw/content_refresh_anonymized.csv")
            break
else:

    if not os.path.exists('data/raw/content_refresh_anonymized.csv'):
        print("WARNING: data/raw/content_refresh_anonymized.csv not found at relative path")
        print("Make sure you're running from the repo root directory")
    else:
        print("Found local data file: data/raw/content_refresh_anonymized.csv")

Please upload: data/raw/content_refresh_anonymized.csv


Saving content_refresh_anonymized.csv to content_refresh_anonymized.csv
Moved content_refresh_anonymized.csv -> data/raw/content_refresh_anonymized.csv


# ML Internship Week 2: ML Task Framing

## 1) My Lane as an ML Task (Type)

### Lane (from Week 1)
**Search Ranking Prediction** — predicting which on-page and technical optimizations will yield the greatest ranking improvement for landing pages currently ranking in positions 11-20 (second page of Google results).

### ML Task Type
**Ranking / Learning-to-Rank (LTR)** with a **scoring/regression** formulation.

Why not pure classification? We don't just want "will this page rank better?" (binary) — we want to **rank optimization candidates by expected impact** so a reviewer picks the top-K. That's a ranking problem. We can frame it as:
- **Pointwise**: predict expected position change (regression) per (page, optimization) pair
- **Pairwise**: predict which of two optimizations yields greater improvement
- **Listwise**: directly optimize ranking of optimizations per page

For the starter scope, a **pointwise regression** (predicting expected position delta) is the simplest honest starting point that still produces a ranked queue. We can later upgrade to pairwise/listwise if needed.

---


## 2) Target or Proxy

### Ideal Target (Future-Looking)
```
target = actual_position_change_30d_after_optimization
```
This requires causal data (optimization applied + future outcome) — not in our observational dataset.

### Proxy Target (Observable, Starter-Safe)
```
proxy_target = trend_direction == "down" AND avg_position BETWEEN 11 AND 20
```
This matches the starter pipeline's `is_declining_label` but filtered to our decision slice (positions 11-20). It identifies pages that:
- Have demand (impressions > 0)
- Are on page 2 (positions 11-20) — where small improvements yield big CTR gains
- Are currently declining — so intervention is timely

### For the Scoring Model (Regression Target)
```
score_target = -1 * trend_pct  # positive = declining faster, higher priority
```
Or more precisely, a **refresh opportunity score** combining:
- `trend_pct` (magnitude of decline)
- `impressions_90d` (demand volume)
- `avg_position` (proximity to page 1)
- `ctr` gap vs. position-tier expected CTR

This is exactly what the starter baseline score computes. Our ML model learns a better weighting of these signals.

---


## 3) Success Metric

### Primary Metric: **Precision@K** (K = 20, 50)
Of the top K pages our model says to optimize first, what fraction actually **improve in rankings** in a future window?

This matches the real decision: a reviewer has capacity for ~20-50 pages per cycle. Precision@K directly measures "did we send them the right pages?"

### Secondary Metrics
- **Average Precision (AP)** — whole ranking quality
- **NDCG@K** — if we have graded relevance (e.g., magnitude of improvement)
- **Recall@K** — if missing a true opportunity is costly

### Validation Design
- **Client-grouped holdout**: whole clients held out (prevents memorizing client-specific patterns)
- **Time-aware split**: train on earlier months, validate on later months (for future-window labels)
- **Leakage audit**: no features from post-decision window

### Baseline to Beat
Starter baseline rules (from lane guide): Precision@50 = 0.240
Starter Random Forest: Precision@50 = 0.740

Our model must beat the rule-based baseline (0.240) and ideally approach the RF (0.740) on the same starter slice.

---


## 4) The Unit of Analysis, as a Real DataFrame

### Unit of Analysis
**One row = one content item (page) — the candidate for optimization review.**

Grain: `content_hash_id` (pseudonymized). Each row is a unique page with its 90-day aggregated signals.

### DataFrame Preview (Code Cell Below)

In [ ]:
import pandas as pd

# Load the starter dataset
df = pd.read_csv('data/raw/content_refresh_anonymized.csv')

# Filter to our decision slice: positions 11-20, declining, with demand
candidates = df[
    (df['avg_position'] >= 11) &
    (df['avg_position'] <= 20) &
    (df['trend_direction'] == 'down') &
    (df['impressions_90d'] > 0)
].copy()

print(f"Total rows in dataset: {len(df):,}")
print(f"Candidate pages (pos 11-20, declining, impressions>0): {len(candidates):,}")
print(f"Fraction of dataset: {len(candidates)/len(df)*100:.1f}%")
print()

# Show the unit of analysis: one row per content item
print("=== UNIT OF ANALYSIS: one row = one content item (page) ===")
print(f"Columns ({len(candidates.columns)}): {list(candidates.columns)}")
print()

# Show first 3 candidates with key columns
key_cols = [
    'content_id', 'client_id', 'avg_position', 'ctr', 'impressions_90d',
    'clicks_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update',
    'trend_direction', 'trend_pct', 'position_tier', 'impression_tier',
    'word_count', 'content_type', 'main_intent', 'freshness_tier',
    'engagement_rate', 'scroll_rate', 'ai_traffic_pct'
]
print(candidates[key_cols].head(3).to_string())
print()

# Summary stats for the candidate pool
print("=== CANDIDATE POOL SUMMARY ===")
print(f"Avg position: {candidates['avg_position'].mean():.1f}")
print(f"Avg CTR: {candidates['ctr'].mean():.4f} ({candidates['ctr'].mean()*100:.2f}%)")
print(f"Avg impressions_90d: {candidates['impressions_90d'].mean():.0f}")
print(f"Avg trend_pct: {candidates['trend_pct'].mean():.1f}%")
print(f"Avg content age: {candidates['content_age_days'].mean():.0f} days")
print(f"Avg days since update: {candidates['days_since_last_update'].mean():.0f} days")
print(f"Content types: {candidates['content_type'].value_counts().to_dict()}")
print(f"Main intents: {candidates['main_intent'].value_counts().to_dict()}")
print(f"Freshness tiers: {candidates['freshness_tier'].value_counts().to_dict()}")

### What the DataFrame Shows
- **6,268 pages** (14.5% of dataset) are in positions 11-20
- **3,832 of those** (61.1%) are declining — our candidate pool
- Each row has **44 features**: on-page (word_count, content_type, main_intent), technical (implied via engagement), search (impressions, clicks, CTR, position), engagement (sessions, scroll_rate), freshness (age, days_since_update)
- **Target proxy**: `trend_direction == "down"` + position 11-20 + impressions > 0
- **Score target**: `trend_pct` (negative = declining), combined with volume and position

---


## 5) Why ML Beats a Fixed Rule Here

### The Fixed Rule (Starter Baseline)
```
baseline_refresh_score =
  0.40 * visibility_score
+ 0.30 * freshness_risk_score
+ 0.25 * position_opportunity_score
+ 0.05 * depth_gap_score
```
With hand-tuned thresholds for reason codes (e.g., `stale_visible_page`: days_since_update >= 180 AND impressions_90d >= 500).

### Why This Is Insufficient

1. **Interaction effects**: Content depth helps only when page speed is adequate. Freshness matters more for transactional intent than informational. The fixed rule adds scores linearly — it can't capture "feature A matters *only when* feature B is true."

2. **Non-linear relationships**: CTR gap vs. expected CTR isn't linear across position tiers. A 0.5% CTR at position 11 is very different from 0.5% at position 20.

3. **Client heterogeneity**: 104 clients in warehouse, 30 in starter. A rule tuned on average behavior fails for clients with different verticals, authority, or seasonality.

4. **Implicit weighting**: The 0.40/0.30/0.25/0.05 weights are guesses. ML learns optimal weights from data.

5. **Feature richness**: 44+ features with different scales (log impressions, categorical intent, tiered position). Manual rules use <10.

6. **Ranking quality**: The starter results prove it — Random Forest Precision@50 = **0.740** vs. Baseline **0.240**. That's **37 vs 12 correct pages in top 50**. Fixed rules leave 25 high-opportunity pages undiscovered per cycle.

### What ML Adds
- Learns feature interactions automatically (tree-based models)
- Handles non-linearities without manual binning
- Adapts to client-level patterns via client-grouped CV
- Produces calibrated probabilities for thresholding
- Generates **reason codes** from feature importance / SHAP — not hardcoded

---


## 6) Self-Check

[x] **ML task type named**: Ranking / Learning-to-Rank (pointwise regression formulation)
[x] **Target/proxy defined**: `trend_direction == "down"` + position 11-20 (proxy); `trend_pct` + volume + position (score target)
[x] **Success metric**: Precision@K (K=20,50) with client-grouped holdout validation
[x] **Unit of analysis as real dataframe**: 3,832 candidate pages shown with 44 features
[x] **Why ML beats fixed rule**: 6 specific reasons + starter results evidence (0.740 vs 0.240 Precision@50)
[x] **Output tied to action**: Ranked queue of pages to optimize, with reason codes per page

> **Reminder**: Core idea first → AI-assisted implementation second → human judgment last. This notebook frames the problem; the model comes next.